In [4]:
!nvidia-smi

Thu Feb 19 16:53:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L40                     Off |   00000000:00:10.0 Off |                    0 |
| N/A   37C    P8             35W /  300W |      14MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
import ollama
import pickle
import os
import time

# --- CONFIGURATION ---
FILE_PATH = 'combined_oran_all.txt'
DB_FILE = 'vector_db_oran_all.pkl'
EMBEDDING_MODEL = 'hf.co/CompendiumLabs/bge-base-en-v1.5-gguf'
LANGUAGE_MODEL = 'hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF'
CHUNK_SIZE = 1000  
CHUNK_OVERLAP = 100 

def create_chunks(text, size, overlap):
    chunks = []
    for i in range(0, len(text), size - overlap):
        chunk_content = text[i:i + size]
        metadata = {"source": FILE_PATH, "start_char": i}
        chunks.append({"content": chunk_content, "metadata": metadata})
    return chunks

def cosine_similarity(a, b):
    dot_product = sum([x * y for x, y in zip(a, b)])
    norm_a = sum([x ** 2 for x in a]) ** 0.5
    norm_b = sum([x ** 2 for x in b]) ** 0.5
    return dot_product / (norm_a * norm_b)

# --- 1. LOAD OR BUILD DATABASE ---
VECTOR_DB = []
db_start_time = time.perf_counter()

if os.path.exists(DB_FILE):
    print(f"--- Loading existing database from {DB_FILE} ---")
    with open(DB_FILE, 'rb') as f:
        VECTOR_DB = pickle.load(f)
    db_end_time = time.perf_counter()
    print(f"✅ Loaded {len(VECTOR_DB)} chunks in {db_end_time - db_start_time:.4f} seconds.")
else:
    print(f"--- Database not found. Processing {FILE_PATH} ---")
    if not os.path.exists(FILE_PATH):
        print(f"Error: {FILE_PATH} not found. Please provide the source text file.")
        exit()

    with open(FILE_PATH, 'r', encoding='utf-8') as file:
        raw_text = file.read()
    
    raw_chunks = create_chunks(raw_text, CHUNK_SIZE, CHUNK_OVERLAP)
    total_chunks = len(raw_chunks)
    print(f"Total chunks to process: {total_chunks}\n")

    for i, item in enumerate(raw_chunks):
        chunk_start = time.perf_counter() # Start timing for THIS specific chunk
        
        try:
            response = ollama.embed(model=EMBEDDING_MODEL, input=item['content'], truncate=True)
            embedding = response['embeddings'][0]
            
            VECTOR_DB.append({
                "content": item['content'],
                "metadata": item['metadata'],
                "embedding": embedding
            })
            
            chunk_end = time.perf_counter()
            chunk_duration = chunk_end - chunk_start
            
            # --- CHUNK LOGGING ---
            progress_pct = ((i + 1) / total_chunks) * 100
            print(f"[{i+1}/{total_chunks}] {progress_pct:.1f}% | Latency: {chunk_duration:.4f}s")
            
        except Exception as e:
            print(f"❌ Error at chunk {i}: {e}")

    # Save to disk
    with open(DB_FILE, 'wb') as f:
        pickle.dump(VECTOR_DB, f)
    
    db_end_time = time.perf_counter()
    print(f"\n✅ Total indexing completed in {db_end_time - db_start_time:.4f} seconds.")

# --- 2. RETRIEVAL ---
def retrieve(query, top_n=3):
    start_retrieval = time.perf_counter()
    
    query_emb = ollama.embed(model=EMBEDDING_MODEL, input=query)['embeddings'][0]
    similarities = []
    
    for item in VECTOR_DB:
        score = cosine_similarity(query_emb, item['embedding'])
        similarities.append((item, score))
    
    similarities.sort(key=lambda x: x[1], reverse=True)
    results = similarities[:top_n]
    
    end_retrieval = time.perf_counter()
    print(f"🔍 Retrieval Phase took: {end_retrieval - start_retrieval:.4f} seconds.")
    return results

# --- 3. EXECUTION ---
input_query = input('\nAsk me a question: ')
retrieved_results = retrieve(input_query)

# Prepare context for the prompt
context_text = "\n".join([f"- {item['content']}" for item, score in retrieved_results])

instruction_prompt = f'''You are a helpful chatbot. Use the context below to answer.
Context:
{context_text}
'''

print('\n--- Context Found ---')
for item, score in retrieved_results:
    print(f" - [Score: {score:.2f}] (Offset: {item['metadata']['start_char']})")

print('\nChatbot response:')
start_gen = time.perf_counter()
first_token_received = False
ttft = 0.0

stream = ollama.chat(
    model=LANGUAGE_MODEL,
    messages=[
        {'role': 'system', 'content': instruction_prompt},
        {'role': 'user', 'content': input_query},
    ],
    stream=True,
)

for chunk in stream:
    if not first_token_received:
        ttft = time.perf_counter() - start_gen
        first_token_received = True
    print(chunk['message']['content'], end='', flush=True)

end_gen = time.perf_counter()
print(f"\n\n🤖 Generation Phase took: {end_gen - start_gen:.4f} seconds.")
print(f"⚡ Time to First Token (TTFT): {ttft:.4f} seconds.")

--- Database not found. Processing combined_oran_all.txt ---
Total chunks to process: 17141

[1/17141] 0.0% | Latency: 0.0424s
[2/17141] 0.0% | Latency: 0.0217s
[3/17141] 0.0% | Latency: 0.0220s
[4/17141] 0.0% | Latency: 0.0291s
[5/17141] 0.0% | Latency: 0.0240s
[6/17141] 0.0% | Latency: 0.0241s
[7/17141] 0.0% | Latency: 0.0221s
[8/17141] 0.0% | Latency: 0.0242s
[9/17141] 0.1% | Latency: 0.0237s
[10/17141] 0.1% | Latency: 0.0215s
[11/17141] 0.1% | Latency: 0.0268s
[12/17141] 0.1% | Latency: 0.0232s
[13/17141] 0.1% | Latency: 0.0212s
[14/17141] 0.1% | Latency: 0.0212s
[15/17141] 0.1% | Latency: 0.0234s
[16/17141] 0.1% | Latency: 0.0234s
[17/17141] 0.1% | Latency: 0.0226s
[18/17141] 0.1% | Latency: 0.0226s
[19/17141] 0.1% | Latency: 0.0220s
[20/17141] 0.1% | Latency: 0.0193s
[21/17141] 0.1% | Latency: 0.0215s
[22/17141] 0.1% | Latency: 0.0225s
[23/17141] 0.1% | Latency: 0.0233s
[24/17141] 0.1% | Latency: 0.0212s
[25/17141] 0.1% | Latency: 0.0239s
[26/17141] 0.2% | Latency: 0.0252s
[27/17

In [7]:
import ollama
import pickle
import os
import time  # New import for timing

# --- CONFIGURATION ---
FILE_PATH = 'combined_oran_all.txt'
DB_FILE = 'vector_db_oran_all.pkl'
EMBEDDING_MODEL = 'hf.co/CompendiumLabs/bge-base-en-v1.5-gguf'
LANGUAGE_MODEL = 'hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF'
CHUNK_SIZE = 1000  
CHUNK_OVERLAP = 100 

def create_chunks(text, size, overlap):
    chunks = []
    for i in range(0, len(text), size - overlap):
        chunk_content = text[i:i + size]
        metadata = {"source": FILE_PATH, "start_char": i}
        chunks.append({"content": chunk_content, "metadata": metadata})
    return chunks

def cosine_similarity(a, b):
    dot_product = sum([x * y for x, y in zip(a, b)])
    norm_a = sum([x ** 2 for x in a]) ** 0.5
    norm_b = sum([x ** 2 for x in b]) ** 0.5
    return dot_product / (norm_a * norm_b)

# --- 1. LOAD OR BUILD DATABASE (WITH LATENCY LOGGING) ---
VECTOR_DB = []
db_start_time = time.perf_counter()

if os.path.exists(DB_FILE):
    print(f"--- Loading existing database from {DB_FILE} ---")
    with open(DB_FILE, 'rb') as f:
        VECTOR_DB = pickle.load(f)
    db_end_time = time.perf_counter()
    print(f"✅ Loaded {len(VECTOR_DB)} chunks in {db_end_time - db_start_time:.4f} seconds.")
else:
    print(f"--- Database not found. Processing {FILE_PATH} ---")
    with open(FILE_PATH, 'r', encoding='utf-8') as file:
        raw_text = file.read()
    
    raw_chunks = create_chunks(raw_text, CHUNK_SIZE, CHUNK_OVERLAP)
    
    for i, item in enumerate(raw_chunks):
        try:
            response = ollama.embed(model=EMBEDDING_MODEL, input=item['content'], truncate=True)
            embedding = response['embeddings'][0]
            VECTOR_DB.append({
                "content": item['content'],
                "metadata": item['metadata'],
                "embedding": embedding
            })
        except Exception as e:
            print(f"Error at chunk {i}: {e}")

    with open(DB_FILE, 'wb') as f:
        pickle.dump(VECTOR_DB, f)
    
    db_end_time = time.perf_counter()
    print(f"✅ Created and saved database in {db_end_time - db_start_time:.4f} seconds.")

# --- 2. RETRIEVAL (WITH LATENCY LOGGING) ---
def retrieve(query, top_n=3):
    start_retrieval = time.perf_counter()
    
    query_emb = ollama.embed(model=EMBEDDING_MODEL, input=query)['embeddings'][0]
    similarities = []
    for item in VECTOR_DB:
        score = cosine_similarity(query_emb, item['embedding'])
        similarities.append((item, score))
    
    similarities.sort(key=lambda x: x[1], reverse=True)
    results = similarities[:top_n]
    
    end_retrieval = time.perf_counter()
    print(f"🔍 Retrieval Phase took: {end_retrieval - start_retrieval:.4f} seconds.")
    return results

# --- 3. GENERATION (WITH LATENCY LOGGING) ---
input_query = input('\nAsk me a question: ')
retrieved_results = retrieve(input_query)

context_text = "\n".join([f"- {item['content']}" for item, score in retrieved_results])

instruction_prompt = f'''You are a helpful chatbot. Use the context below to answer.
Context:
{context_text}
'''

print('\n--- Context Found ---')
context_text = ""
for item, score in retrieved_results:
    print(f" - [Score: {score:.2f}] (Offset: {item['metadata']['start_char']})")
    print(item)
    context_text += f"\n- {item['content']}"
    

print('\nChatbot response:')
start_gen = time.perf_counter()
first_token_received = False

stream = ollama.chat(
    model=LANGUAGE_MODEL,
    messages=[
        {'role': 'system', 'content': instruction_prompt},
        {'role': 'user', 'content': input_query},
    ],
    stream=True,
)

for chunk in stream:
    if not first_token_received:
        # Measure time to first token (TTFT)
        ttft = time.perf_counter() - start_gen
        first_token_received = True
    print(chunk['message']['content'], end='', flush=True)

end_gen = time.perf_counter()
print(f"\n\n🤖 Generation Phase took: {end_gen - start_gen:.4f} seconds.")
print(f"⚡ Time to First Token (TTFT): {ttft:.4f} seconds.")



--- Loading existing database from vector_db_oran_all.pkl ---
✅ Loaded 17138 chunks in 0.5468 seconds.
🔍 Retrieval Phase took: 2.4274 seconds.

--- Context Found ---
 - [Score: 0.76] (Offset: 4046400)
{'content': 'ortability would be beneficial, the design choices of the R1 interface should not unnecessarily become an obstacle for such portability. \uf0b7 Leverage Existing Work: Because rApps will share some common functionality with xApps (startup, messaging, data sharing, etc.), the prior work done for xApps should be analyzed for applicability to the R1 use cases and leveraged where appropriate. \uf0b7 Aligned integration services: Because integration or enablement services (such as registration, discovery, authorization, authentication, etc.) are needed to integrate rApps with the Non-RT RIC as well as to integrate xApps with the Near-RT RIC, the work done for xApp integration/enablement should be analyzed for applicability to rApp integration/enablement and leveraged where appropr